# Explainable AI (XAI) for Pneumonia Classification
## Gradient-weighted Class Activation Mapping (Grad-CAM)

This notebook loads the trained `best_pneumonia_model.keras` and generates a **Grad-CAM** heatmap for an image. 
Grad-CAM helps us see **which parts of the X-ray the model looked at** to make its prediction.


In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from tensorflow.keras.preprocessing import image

print(f"TensorFlow version: {tf.__version__}")

## 1. Load the Trained Model
We load the `.keras` model that was saved during training.


In [ ]:
MODEL_PATH = 'best_pneumonia_model.keras'

if not os.path.exists(MODEL_PATH):
    print(f"Error: {MODEL_PATH} not found. Please run the training notebook first!")
else:
    model = tf.keras.models.load_model(MODEL_PATH)
    model.summary()

## 2. Grad-CAM Algorithm
We will extract the gradients of the top predicted class relative to the feature map activations of the last convolutional layer.
For `DenseNet121`, the last convolutional block is usually named `conv5_block16_concat` or `relu`. We can automatically find it.


In [ ]:
def get_last_conv_layer_name(model):
    # For DenseNet specifically, the final feature maps come from 'relu' or 'conv5_block16_concat'
    for layer_name in ['relu', 'conv5_block16_concat', 'top_conv', 'top_activation']:
        try:
            if model.get_layer(layer_name):
                return layer_name
        except Exception:
            pass
            
    # Generic fallback: just grab the very last convolutional or concatenate layer
    for layer in reversed(model.layers):
        if 'conv' in layer.name.lower() or 'concat' in layer.name.lower():
            return layer.name
            
    raise ValueError("Could not find 4D layer. Cannot apply GradCAM.")

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    # If the model has a nested base model (like our Transfer Learning setup), we need to adapt.
    # Our model structure: Input -> Data Augmentation -> Preprocess -> base_model -> ... -> Dense
    
    # We create a model that maps the input image to the activations of the last conv layer as well as the output predictions
    base_model = model.layers[3] # Index 3 is typically the base_model (DenseNet121) in our architecture
    
    grad_model = tf.keras.models.Model(
        [base_model.inputs], [base_model.get_layer(last_conv_layer_name).output, base_model.output]
    )

    # However, our main model has preprocessing before the base_model. Let's do preprocessing manually.
    # Or, we can build a new model from model.inputs to [last_conv_layer.output, model.output].
    # But because base_model is a nested layer, tf.GradientTape has trouble seeing inside it unless we flatten or explicitly target the base.
    
    # Let's use a simpler approach: trace through the exact layers of our trained model.
    # Our actual model layout:
    # 0: InputLayer
    # 1: Sequential (Data Aug)
    # 2: TFOpLambda (preprocess_input)
    # 3: Functional (DenseNet121)
    # 4: GlobalAveragePooling2D
    # 5: Dropout
    # 6: Dense
    pass

Let's rewrite a robust Grad-CAM that can handle nested models.


In [ ]:
def generate_gradcam(img_array, full_model, class_index=0):
    # 1. Identity the base model and classifier layers dynamically
    base_model_index = -1
    base_model = None
    for i, layer in enumerate(full_model.layers):
        if 'densenet' in layer.name.lower() or (isinstance(layer, tf.keras.Model) and not isinstance(layer, tf.keras.Sequential)):
            base_model = layer
            base_model_index = i
            break
            
    if base_model is None:
        raise ValueError("Could not find DenseNet base model in the architecture.")
        
    last_conv_layer_name = get_last_conv_layer_name(base_model)
    print(f"Using {last_conv_layer_name} as the last conv layer for Grad-CAM.")
    
    # Create a sub-model that goes from base_model input to base_model's last conv layer & base_model output
    grad_model = tf.keras.models.Model(
        inputs=base_model.inputs,
        outputs=[base_model.get_layer(last_conv_layer_name).output, base_model.output]
    )
    
    # 2. We also need to map base_model output through the final classification head (GAP -> Dropout -> Dense)
    # CRITICAL FIX: We must strip the final sigmoid activation to compute gradients on pure Logits.
    # Otherwise, highly confident predictions (>99% or <1%) will suffer from vanishing gradients 
    # and result in completely blank (blue) heatmaps!
    classifier_input = tf.keras.Input(shape=base_model.output.shape[1:])
    x = classifier_input
    for layer_index in range(base_model_index + 1, len(full_model.layers)):
        layer = full_model.layers[layer_index]
        if isinstance(layer, tf.keras.layers.Dense) and layer_index == len(full_model.layers) - 1:
            # Recreate the dense layer without any activation function (linear)
            linear_layer = tf.keras.layers.Dense(layer.units, activation=None)
            x = linear_layer(x)
            linear_layer.set_weights(layer.get_weights()) # Copy trained weights over
        else:
            x = layer(x)
            
    classifier_model = tf.keras.Model(classifier_input, x)
    
    # 3. Compute Gradients
    with tf.GradientTape() as tape:
        # Get conv outputs and base_model predictions
        conv_outputs, base_preds = grad_model(img_array)
        tape.watch(conv_outputs)
        
        # Get final PRE-SIGMOID predictions (pure logits)
        preds = classifier_model(base_preds)
        
        # We target the specific class (0 for Pneumonia normally if using 1-output unit)
        top_class_channel = preds[:, class_index]

    # Gradient of the pure logits class with respect to the output feature map of the last conv layer
    grads = tape.gradient(top_class_channel, conv_outputs)
    
    # Global average pooling on the gradients over spatial dimensions
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    # Multiply each channel in the feature map by "how important this channel is" (the pooled gradients)
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    
    # Normalize the heatmap safely (prevent division by zero)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()

## 3. Visualize the Heatmap
We need a function to overlay the heatmap on top of the original X-Ray image.


In [ ]:
import matplotlib.patches as patches

def display_gradcam(img_path, heatmap, alpha=0.4, bbox=None):
    # Load original image
    img = image.load_img(img_path)
    img = image.img_to_array(img)
    
    # Rescale heatmap to 0-255
    heatmap = np.uint8(255 * heatmap)
    
    # Use jet colormap
    jet = cm.get_cmap("jet")
    
    # Use RGB values of the colormap
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap]
    
    # Create image with RGB heatmap
    jet_heatmap = image.array_to_img(jet_heatmap)
    jet_heatmap = jet_heatmap.resize((img.shape[1], img.shape[0]))
    jet_heatmap = image.img_to_array(jet_heatmap)
    
    # Superimpose heatmap
    superimposed_img = jet_heatmap * alpha + img
    superimposed_img = image.array_to_img(superimposed_img)
    
    # Display
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    
    ax[0].set_title("Original X-Ray (Ground Truth BBox)")
    ax[0].imshow(image.load_img(img_path), cmap='gray')
    
    if bbox:
        # Drawing GT bbox on both
        # bbox [x,y,w,h]
        rect = patches.Rectangle((bbox[0], bbox[1]), bbox[2], bbox[3], linewidth=2, edgecolor='r', facecolor='none', label='Ground Truth')
        ax[0].add_patch(rect)
        rect2 = patches.Rectangle((bbox[0], bbox[1]), bbox[2], bbox[3], linewidth=2, edgecolor='white', facecolor='none', label='Ground Truth')
        ax[1].add_patch(rect2)
        ax[0].legend()
        ax[1].legend()

    ax[0].axis("off")
    
    ax[1].set_title("Grad-CAM (Attention Map)")
    ax[1].imshow(superimposed_img)
    ax[1].axis("off")
    
    plt.tight_layout()
    plt.show()

## 4. Run Explainable AI on a Sample Image
Let's select an image from the test set. Change `img_path` to test different cases.


In [ ]:
# 4. Load constants and identify base model index globally
base_model_index = -1
for i, layer in enumerate(model.layers):
    if 'densenet' in layer.name.lower() or (isinstance(layer, tf.keras.Model) and not isinstance(layer, tf.keras.Sequential)):
        base_model_index = i
        break

# 4.1 Test on specific uploaded image: 00011136_002.png
bbox_img_path = "C:/Users/prakh/Downloads/archive (1)/chest_xray/00011136_002.png"
# Coordinates from CSV: 574.00, 574.90, 229.83, 166.11
gt_bbox = [574.00, 574.90, 229.83, 166.11] 

if os.path.exists(bbox_img_path):
    print(f"\nTesting on specific BBox image: {bbox_img_path}")
    img = image.load_img(bbox_img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    
    # Use a Keras Submodel for flawless execution of intermediate preprocessing layers
    preprocess_model = tf.keras.Model(inputs=model.inputs, outputs=model.layers[base_model_index - 1].output)
    processed_img = preprocess_model(img_array, training=False)
    
    try:
        heatmap = generate_gradcam(processed_img, model)
        display_gradcam(bbox_img_path, heatmap, bbox=gt_bbox)
    except Exception as e:
        print("Error on BBox image:", e)
else:
    print(f"BBox image not found at {bbox_img_path}")

# 4.2 Search and test regular test images
test_pneumonia_dir = "C:/Users/prakh/Downloads/archive (1)/chest_xray/test/PNEUMONIA"
if os.path.exists(test_pneumonia_dir):
    sample_img_name = os.listdir(test_pneumonia_dir)[0]
    img_path = os.path.join(test_pneumonia_dir, sample_img_name)
    
    print(f"\nTesting on standard pneumonia image: {img_path}")
    
    # ... rest of the preparation logic ...
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    
    # Use a Keras Submodel for flawless execution of intermediate preprocessing layers
    preprocess_model = tf.keras.Model(inputs=model.inputs, outputs=model.layers[base_model_index - 1].output)
    processed_img = preprocess_model(img_array, training=False)
        
    try:
        heatmap = generate_gradcam(processed_img, model)
        display_gradcam(img_path, heatmap)
    except Exception as e:
        print("Model needs to be trained first or there is an architectural mismatch.", e)
else:
    print("Test directory for PNEUMONIA not found.")

## 5. Experimentation: Correcting BBox Scaling for Resized Images
The dataset bounding boxes are provided for the original NIH 1024x1024 images. However, the uploaded image `00011136_002.png` is already pre-resized to 224x224! Drawing the original 500+ coordinates on a 224x224 image expands the plot axes massively and visually shrinks the X-ray, making it look poorly resized. 

Let's scale the BBox by `224 / 1024` to fit the new aspect ratio perfectly.


In [ ]:
# 5.1 Scale the BBox and Re-Draw
original_img_size = 1024.0
current_img_size = 224.0
scale_factor = current_img_size / original_img_size

# gt_bbox = [x, y, w, h] from previously
scaled_bbox = [val * scale_factor for val in gt_bbox]

print(f"Original BBox: {gt_bbox}")
print(f"Scaled BBox for 224x224: {scaled_bbox}")

if os.path.exists(bbox_img_path):
    print("\nVisualizing Grad-CAM with Correctly Scaled BBox Ground Truth:")
    
    # We'll regenerate the heatmap for the specific image just to be completely safe in this cell
    try:
        img = image.load_img(bbox_img_path, target_size=(224, 224))
        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        
        # Use a Keras Submodel for flawless execution of intermediate preprocessing layers
        preprocess_model = tf.keras.Model(inputs=model.inputs, outputs=model.layers[base_model_index - 1].output)
        processed_img = preprocess_model(img_array, training=False)
                    
        heatmap_experiment = generate_gradcam(processed_img, model)
        display_gradcam(bbox_img_path, heatmap_experiment, bbox=scaled_bbox)
    except Exception as e:
        print("Error during specialized scaled visualization:", e)
else:
    print("Image not found.")